In [23]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from colorama import Fore

# Preprocessing Essentials
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
# Google nach: für Standard Scaler formula

# Regressors
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.svm import SVR

# Evaluation Metrics
from sklearn.metrics._regression import mean_absolute_error, mean_squared_error, root_mean_squared_error

In [24]:
os.listdir()

['artifacts',
 'data',
 'EDA.ipynb',
 'machine-learning_2.ipynb',
 'machine_learning_1.ipynb',
 'pre-processing.ipynb',
 'Task.py',
 'Virtualazation.ipynb']

In [25]:
df = pd.read_csv('../ml_Session_23_Task_Heart-Disease-Patients-Records/data/preprocessed-data/preprocessed-data.csv')

In [26]:
df.sample(3)

,age,chest_pain_type,resisting_blood_pressure,cholesterol_level,max_heart_rate_achieved,st_depression,num_major_vessels,thalassemia,sex_female,sex_male,...,rest_ecg_ST-T wave abnormality,rest_ecg_left ventricular hypertrophy,rest_ecg_normal,diagnosis_Diagnosed,diagnosis_Un-Diagnosed,st_slope_downsloping,st_slope_flat,st_slope_upsloping,exercise_induced_angina_no,exercise_induced_angina_yes
206,59,2,126,218,134,2.2,1,0,0,1,...,1,0,0,1,0,0,1,0,1,0
587,61,3,145,307,146,1.0,0,2,1,0,...,0,0,1,1,0,0,1,0,0,1
553,58,3,100,234,156,0.1,1,2,0,1,...,1,0,0,1,0,1,0,0,1,0


# Split features into X and y

## Die sicherere Lösung für deinen Code:
Damit dein Modell wirklich medizinisch lernt, solltest du beide Diagnose-Spalten aus X entfernen:


In [27]:
# Wir löschen BEIDE Diagnose-Spalten aus X
X = df.drop( columns= ['diagnosis_Diagnosed', 'diagnosis_Un-Diagnosed'])

# Und setzen eine davon als unser Ziel y
y = df["diagnosis_Diagnosed"]

In [28]:
X.shape

(1024, 20)

# Split into training and testing

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state= 42)

In [30]:
X_train

,age,chest_pain_type,resisting_blood_pressure,cholesterol_level,max_heart_rate_achieved,st_depression,num_major_vessels,thalassemia,sex_female,sex_male,fasting_blood_sugar_higher than 120mg/ml,fasting_blood_sugar_lower than 120mg/ml,rest_ecg_ST-T wave abnormality,rest_ecg_left ventricular hypertrophy,rest_ecg_normal,st_slope_downsloping,st_slope_flat,st_slope_upsloping,exercise_induced_angina_no,exercise_induced_angina_yes
137,64,3,180,325,154,0.0,0,1,1,0,0,1,1,0,0,1,0,0,0,1
377,67,3,120,237,71,1.0,0,1,0,1,0,1,1,0,0,0,1,0,1,0
388,63,0,145,233,150,2.3,0,0,0,1,1,0,0,0,1,0,0,1,1,0
824,63,2,135,252,172,0.0,0,1,1,0,0,1,0,0,1,1,0,0,1,0
767,46,1,101,197,156,0.0,0,2,0,1,1,0,1,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,51,3,140,299,173,1.6,0,2,0,1,0,1,1,0,0,1,0,0,0,1
270,43,3,110,211,161,0.0,0,2,0,1,0,1,1,0,0,1,0,0,1,0
860,64,2,140,335,158,0.0,0,1,0,1,0,1,1,0,0,1,0,0,1,0
435,59,3,174,249,143,0.0,0,1,1,0,0,1,1,0,0,0,1,0,0,1


# Model Selection
## Getting Your Preproccing Pipeline ready !

In [31]:
df.columns

Index(['age', 'chest_pain_type', 'resisting_blood_pressure',
       'cholesterol_level', 'max_heart_rate_achieved', 'st_depression',
       'num_major_vessels', 'thalassemia', 'sex_female', 'sex_male',
       'fasting_blood_sugar_higher than 120mg/ml',
       'fasting_blood_sugar_lower than 120mg/ml',
       'rest_ecg_ST-T wave abnormality',
       'rest_ecg_left ventricular hypertrophy', 'rest_ecg_normal',
       'diagnosis_Diagnosed', 'diagnosis_Un-Diagnosed', 'st_slope_downsloping',
       'st_slope_flat', 'st_slope_upsloping', 'exercise_induced_angina_no',
       'exercise_induced_angina_yes'],
      dtype='object')

In [32]:
preprocessing_step = ColumnTransformer([
    ( 'One-Hot Encoding', OneHotEncoder(), [ 'sex','fasting_blood_sugar', 'rest_ecg', 'diagnosis', 'st_slope', 'exercise_induced_angina' ]),

], remainder = 'passthtrough' )

preprocessing_step

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('One-Hot Encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthtrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``.

In [33]:
knn_model = KNeighborsRegressor( n_neighbors= 7  )

In [37]:
knn_Pipeline = Pipeline([
    ( 'One-Hor Encoding', preprocessing_step),
    (  'model', knn_model )
])

knn_Pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('One-Hor Encoding', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('One-Hot Encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthtrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

Define a function to quickly evaluate your regressor !

In [35]:
def evaluate_regressor ( model, model_name: str , X_train, X_test, y_train, y_test):

    print( 'Training'+ 35 * "_" + '\n' )

    print( model_name + ':')
    print( '_' * 40 )

    model.fit( X_train, y_train )
    y_pred_train = model.predict ( X_train )
    y_pred_test = model.predict( X_test )

    print( )

    MAE_train = mean_absolute_error ( y_train, y_pred_train )
    MSE_train = mean_squared_error ( y_train, y_pred_train )
    RMSE_train = root_mean_squared_error( y_train, y_pred_test )

    MAE_test = mean_absolute_error ( y_test, y_pred_test )
    MSE_test = mean_squared_error ( y_test, y_pred_test )
    RMSE_test = root_mean_squared_error ( y_test, y_pred_test )

    print( f'Mean absolute error of training: {MAE_train}' )
    print( Fore.CYAN + f'Mean squared error of training: {MSE_train}' )
    print( Fore.WHITE + f'Root mean squared error of training: {RMSE_train}' )
    print( )

    print( Fore.WHITE + f'Mean absolute error of testing: {MAE_test}' )
    print( Fore.CYAN + f'Mean squared error of testing: {MSE_test}' )
    print( Fore.WHITE + f'Root mean squared error of testing: {RMSE_test}' )
    

## Pre-Processing

In [38]:
knn_Pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('One-Hor Encoding', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('One-Hot Encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthtrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

# Error wegen: daten sind schon als One-Hoting gesetzt

In [41]:
# knn_Pipeline.fit( X_train, y_train )
# Error wegen: daten sind schon als One-Hoting gesetzt